# Activity 1: Machine Learning Classification and Regression
Dataset: Phishing URL dataset (Kaggle) with lexical features. **Classification:** detect phishing URLs. **Regression:** predict the entropy (randomness) of a URL.

Upload `phishing_final_prepared_dataset.csv` to the Colab session (left sidebar, Files) before running, or run the first data cell and use the upload dialog.

In [ ]:
import warnings, os, time; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
                             mean_absolute_error, mean_squared_error, r2_score)
sns.set_style("whitegrid")
SEED = 42
SAMPLE_N = 20000      # rows used for model training/testing (keeps SVM and boosting fast); raise it if you want

## 1. Load the dataset

In [ ]:
CSV = "phishing_final_prepared_dataset.csv"
if not os.path.exists(CSV):
    from google.colab import files
    files.upload()                       # choose phishing_final_prepared_dataset.csv
df = pd.read_csv(CSV)
print("=== DATASET: Phishing URLs ===")
print("Shape:", df.shape)
print("Missing values per column (only columns with missing values):")
print(df.isna().sum()[df.isna().sum() > 0])
print("Duplicate rows:", int(df.duplicated().sum()))
print("Class counts:", df["label"].value_counts().to_dict(), "(1 = phishing, 0 = legitimate)")
display(df.head(3))

## 2. Cleaning
The missing values are only in the text columns `dom` and `tld`, which are not used as model inputs. Duplicates are removed. All numeric lexical features are used; the raw text columns (`url`, `dom`, `tld`) are dropped.

In [ ]:
df = df.drop_duplicates()
TEXT_COLS = ["url", "dom", "tld"]
data = df.drop(columns=TEXT_COLS)
print("After cleaning:", data.shape, "| missing in numeric data:", int(data.isna().sum().sum()))
print("Class balance: %.1f%% phishing" % (100 * data["label"].mean()))

# stratified sample for modelling
samp, _ = train_test_split(data, train_size=SAMPLE_N, stratify=data["label"], random_state=SEED)
samp = samp.reset_index(drop=True)
print("Modelling sample:", samp.shape, "| phishing share: %.1f%%" % (100 * samp["label"].mean()))

# PART A: CLASSIFICATION - Phishing URL detection
## 3. Exploratory data analysis

In [ ]:
names = {0: "legitimate", 1: "phishing"}
plt.figure(figsize=(5, 3.6))
sns.countplot(x=data["label"].map(names), hue=data["label"].map(names), legend=False)
plt.title("Class distribution (full dataset)"); plt.xlabel(""); plt.tight_layout()
plt.savefig("fig_c_class.png", dpi=200); plt.show()

feats = ["url_len", "dot_cnt", "digit_cnt", "subdom_cnt", "path_len", "entropy"]
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for ax, f in zip(axes.ravel(), feats):
    d = samp.assign(lab=samp["label"].map(names), v=np.log1p(samp[f]) if f != "entropy" else samp[f])
    sns.boxplot(data=d, x="lab", y="v", hue="lab", legend=False, ax=ax, showfliers=False)
    ax.set_title(f + (" (log scale)" if f != "entropy" else "")); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.savefig("fig_c_box.png", dpi=200); plt.show()

X_all = samp.drop(columns="label"); y_all = samp["label"]
top = X_all.corrwith(y_all).abs().sort_values(ascending=False).head(10).index
print("Top-10 features by |correlation| with label:")
print(X_all.corrwith(y_all)[top].round(3))
plt.figure(figsize=(7.5, 6)); sns.heatmap(X_all[top].corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 7})
plt.title("Correlation: top-10 features related to the target"); plt.tight_layout()
plt.savefig("fig_c_corr.png", dpi=200); plt.show()

## 4. Train/test split and model comparison (5-fold stratified CV on the training set)

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED)
print("Train:", Xtr.shape, "Test:", Xte.shape, "| phishing share train/test: %.3f / %.3f" % (ytr.mean(), yte.mean()))

models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "SVM (RBF)": SVC(probability=True, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows, fitted = [], {}
for name, m in models.items():
    t0 = time.time()
    pipe = make_pipeline(StandardScaler(), m)
    cv_pipe = make_pipeline(StandardScaler(), SVC(random_state=SEED)) if name == "SVM (RBF)" else pipe   # no Platt scaling needed for CV F1
    cvf1 = cross_val_score(cv_pipe, Xtr, ytr, cv=cv, scoring="f1", n_jobs=-1)
    pipe.fit(Xtr, ytr)
    pred, prob = pipe.predict(Xte), pipe.predict_proba(Xte)[:, 1]
    fitted[name] = (pipe, pred, prob)
    rows.append(dict(Model=name, CV_F1=cvf1.mean(), Accuracy=accuracy_score(yte, pred),
                     Precision=precision_score(yte, pred), Recall=recall_score(yte, pred),
                     F1=f1_score(yte, pred), AUC=roc_auc_score(yte, prob), PR_AUC=average_precision_score(yte, prob)))
    print(f"  {name} done in {time.time() - t0:.1f}s")
res_c = pd.DataFrame(rows).round(4)
print("\n=== CLASSIFICATION RESULTS (test set; phishing = positive class) ===")
print(res_c.to_string(index=False)); res_c.to_csv("res_c.csv", index=False)

### Hyper-parameter tuning (SVM)

In [ ]:
grid = GridSearchCV(make_pipeline(StandardScaler(), SVC(random_state=SEED)),
                    {"svc__C": [1, 10, 100], "svc__gamma": ["scale", 0.01, 0.1]},
                    cv=StratifiedKFold(3, shuffle=True, random_state=SEED), scoring="f1", n_jobs=-1).fit(Xtr, ytr)
print("Best SVM parameters:", grid.best_params_, "| CV F1 = %.4f" % grid.best_score_)
svm_best = make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED,
                         C=grid.best_params_["svc__C"], gamma=grid.best_params_["svc__gamma"])).fit(Xtr, ytr)
p, pr = svm_best.predict(Xte), svm_best.predict_proba(Xte)[:, 1]
tuned = dict(Model="Tuned SVM", CV_F1=grid.best_score_, Accuracy=accuracy_score(yte, p), Precision=precision_score(yte, p),
             Recall=recall_score(yte, p), F1=f1_score(yte, p), AUC=roc_auc_score(yte, pr), PR_AUC=average_precision_score(yte, pr))
fitted["Tuned SVM"] = (svm_best, p, pr)
res_c = pd.concat([res_c, pd.DataFrame([tuned]).round(4)], ignore_index=True)
print(res_c.to_string(index=False)); res_c.to_csv("res_c.csv", index=False)
open("svm_params.txt", "w").write(f"C={grid.best_params_['svc__C']}, gamma={grid.best_params_['svc__gamma']}, cvf1={grid.best_score_:.4f}")

## 5. Confusion matrix, ROC and precision-recall curves, feature importance

In [ ]:
best = res_c.sort_values("CV_F1").iloc[-1]["Model"]        # chosen on cross-validation, not on the test set
print("Best model by CV F1:", best)
cm = confusion_matrix(yte, fitted[best][1]); print(cm)
plt.figure(figsize=(4.4, 3.8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["legitimate", "phishing"], yticklabels=["legitimate", "phishing"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix - {best}"); plt.tight_layout()
plt.savefig("fig_c_cm.png", dpi=200); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for n in fitted:
    f, t, _ = roc_curve(yte, fitted[n][2]); ax[0].plot(f, t, label=f"{n} ({roc_auc_score(yte, fitted[n][2]):.3f})")
    pr_, rc_, _ = precision_recall_curve(yte, fitted[n][2]); ax[1].plot(rc_, pr_, label=f"{n} ({average_precision_score(yte, fitted[n][2]):.3f})")
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curves"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curves"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig("fig_c_roc.png", dpi=200); plt.show()

rf = fitted["Random Forest"][0].named_steps["randomforestclassifier"]
imp = pd.Series(rf.feature_importances_, index=X_all.columns).sort_values(ascending=False).head(10)
print(imp.round(4)); imp.round(4).to_csv("imp_c.csv")
plt.figure(figsize=(6.5, 4)); sns.barplot(x=imp.values, y=imp.index, color="#2e86c1")
plt.title("Random Forest - top 10 feature importances"); plt.tight_layout(); plt.savefig("fig_c_imp.png", dpi=200); plt.show()

## 6. Decision threshold analysis (cost of missed phishing URLs)

In [ ]:
prob = fitted[best][2]
th = pd.DataFrame([{"Threshold": t, "Precision": precision_score(yte, prob >= t), "Recall": recall_score(yte, prob >= t),
                    "F1": f1_score(yte, prob >= t), "Missed phishing": int(((prob < t) & (yte == 1)).sum()),
                    "False alarms": int(((prob >= t) & (yte == 0)).sum())} for t in [0.5, 0.4, 0.3, 0.2, 0.1]]).round(4)
print(th.to_string(index=False)); th.to_csv("thresholds.csv", index=False)

# PART B: REGRESSION - predicting URL entropy
The Shannon **entropy** of a URL measures how random its characters are (random-looking strings give high entropy). We predict it from the other structural features, excluding the label and the entropy itself. Note that entropy can also be computed directly from the raw URL text, so this task is used to study how structural features relate to randomness.

## 7. Exploratory data analysis

In [ ]:
yr = samp["entropy"]; Xr = samp.drop(columns=["entropy", "label"])
print("Target entropy: mean %.3f, std %.3f, min %.3f, max %.3f" % (yr.mean(), yr.std(), yr.min(), yr.max()))
plt.figure(figsize=(5.2, 3.6)); sns.histplot(yr, kde=True, color="#2e86c1")
plt.title("Target: URL entropy"); plt.tight_layout(); plt.savefig("fig_r_hist.png", dpi=200); plt.show()

corr_t = Xr.corrwith(yr).sort_values(ascending=False); print(corr_t.round(3))
cols = list(corr_t.abs().sort_values(ascending=False).head(10).index) + ["entropy"]
plt.figure(figsize=(7.5, 6)); sns.heatmap(samp[cols].corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 7})
plt.title("Correlation matrix (top features and target)"); plt.tight_layout(); plt.savefig("fig_r_corr.png", dpi=200); plt.show()

## 8. Models and results (5-fold CV on the training set)

In [ ]:
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(Xr, yr, test_size=0.2, random_state=SEED)
kf = KFold(5, shuffle=True, random_state=SEED)
reg_models = {"Linear Regression": LinearRegression(), "Ridge": Ridge(alpha=1.0), "Lasso": Lasso(alpha=0.001, max_iter=10000),
              "Random Forest": RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
              "Gradient Boosting": GradientBoostingRegressor(random_state=SEED)}
rows_r, rfit = [], {}
for name, m in reg_models.items():
    pipe = make_pipeline(StandardScaler(), m)
    cvr2 = cross_val_score(pipe, Xtr_r, ytr_r, cv=kf, scoring="r2", n_jobs=-1)
    pipe.fit(Xtr_r, ytr_r); pred = pipe.predict(Xte_r); rfit[name] = (pipe, pred)
    rows_r.append(dict(Model=name, CV_R2=cvr2.mean(), MAE=mean_absolute_error(yte_r, pred),
                       RMSE=np.sqrt(mean_squared_error(yte_r, pred)), R2=r2_score(yte_r, pred)))

g = GridSearchCV(make_pipeline(StandardScaler(), GradientBoostingRegressor(random_state=SEED)),
                 {"gradientboostingregressor__n_estimators": [100, 300],
                  "gradientboostingregressor__learning_rate": [0.05, 0.1],
                  "gradientboostingregressor__max_depth": [2, 3, 5]}, cv=kf, scoring="r2", n_jobs=-1).fit(Xtr_r, ytr_r)
pred = g.predict(Xte_r); rfit["Tuned Gradient Boosting"] = (g, pred)
rows_r.append(dict(Model="Tuned Gradient Boosting", CV_R2=g.best_score_, MAE=mean_absolute_error(yte_r, pred),
                   RMSE=np.sqrt(mean_squared_error(yte_r, pred)), R2=r2_score(yte_r, pred)))
res_r = pd.DataFrame(rows_r).round(4)
print("=== REGRESSION RESULTS (test set) ==="); print(res_r.to_string(index=False)); res_r.to_csv("res_r.csv", index=False)
print("Best tuned parameters:", g.best_params_)
open("gb_params.txt", "w").write(str(g.best_params_))

## 9. Regression plots

In [ ]:
bestr = res_r.sort_values("CV_R2").iloc[-1]["Model"]; pr_ = rfit[bestr][1]
print("Best regression model (by CV R2):", bestr)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(yte_r, pr_, alpha=.4, s=10); ax[0].plot([yte_r.min(), yte_r.max()], [yte_r.min(), yte_r.max()], "r--")
ax[0].set(xlabel="Actual", ylabel="Predicted", title=f"Actual vs Predicted - {bestr}")
sns.histplot(yte_r - pr_, kde=True, ax=ax[1], color="#8e44ad"); ax[1].set_title("Residuals")
plt.tight_layout(); plt.savefig("fig_r_pred.png", dpi=200); plt.show()

plt.figure(figsize=(6, 3.8)); sns.barplot(x=res_r["R2"], y=res_r["Model"], color="#2e86c1")
plt.title("Test R2 by model"); plt.tight_layout(); plt.savefig("fig_r_r2.png", dpi=200); plt.show()

co = pd.Series(rfit["Linear Regression"][0].named_steps["linearregression"].coef_, index=Xr.columns).sort_values()
print(co.round(4)); co.round(4).to_csv("coef_r.csv")
plt.figure(figsize=(6, 5.5)); sns.barplot(x=co.values, y=co.index, color="#16a085")
plt.title("Linear regression coefficients (standardised)"); plt.tight_layout(); plt.savefig("fig_r_coef.png", dpi=200); plt.show()

rfr = rfit["Random Forest"][0].named_steps["randomforestregressor"]
impr = pd.Series(rfr.feature_importances_, index=Xr.columns).sort_values(ascending=False).head(8)
print(impr.round(4)); impr.round(4).to_csv("imp_r.csv")